# VOREKO IA — Actividad 5: Preparación de datos, entrenamiento y registro en MLflow

**Master de Inteligencia Artificial**  
José Ignacio Castillo Camacho

## 1. Entorno

In [ ]:
import sys, os, subprocess, importlib, json, platform, warnings
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    %pip install -q "mlflow>=3.0" "xgboost>=2.0" openpyxl
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
from importlib.metadata import version
for p in ["pandas", "numpy", "scikit-learn", "xgboost", "scipy", "mlflow", "matplotlib", "openpyxl"]:
    print(f"{p:<14} {version(p)}")
print("Python", platform.python_version(), "| Entorno:", "Google Colab" if EN_COLAB else "local", "| CPU:", os.cpu_count())

## 2. Repositorio y rutas

In [ ]:
from pathlib import Path
URL_REPO = "https://github.com/Ignacio-Castillo-C/voreko-ia-incidencias.git"
if EN_COLAB and not Path("fuentes/datos_prep.py").exists():
    if not Path("repo").exists():
        subprocess.run(["git", "clone", "-q", URL_REPO, "repo"], check=True)
    os.chdir("repo/Actividad5" if Path("repo/Actividad5").exists() else "repo")
RAIZ = Path.cwd().parent if Path.cwd().name == "fuentes" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "fuentes"))
import datos_prep as dp, visualizaciones as vis, train
for m in (dp, vis, train):
    importlib.reload(m)
print("Raíz del proyecto:", RAIZ)

## 3. Datos originales (versión 0)

In [ ]:
ruta = dp.obtener_datos()
crudo = dp.leer_original(ruta)
huella = dp.sha256_dataframe(crudo)
print("Dimensiones con encabezados:", crudo.shape)
print("SHA-256 del contenido:", huella)
assert huella == "7e736374a647e1f64f930dab167dcf93bf8018c9d17b8a5891e5ce257b4671cd", "El archivo no coincide con la versión 0"
display(crudo.iloc[:4, :14])

## 4. Limpieza y estandarización

In [ ]:
limpio, reporte = dp.limpiar(crudo)
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 120)
for paso, info in reporte.items():
    print(f"── {paso}"); print(json.dumps(info, ensure_ascii=False, default=str, indent=1)[:900])

In [ ]:
base, _ = dp.convertir_tipos(dp.aplanar_encabezados(crudo)[0])
sucio = pd.concat([base, base.iloc[[0, 1]], base.iloc[[2]].assign(precio_venta_real_m2=1)], ignore_index=True)
_, rep_dup = dp.deduplicar(sucio)
print({k: rep_dup[k] for k in ["filas_iniciales", "duplicados_exactos", "duplicados_por_clave_proyecto", "filas_finales"]})
assert rep_dup["filas_finales"] == len(base)

In [ ]:
manifiesto = dp.ejecutar()
print(json.dumps({k: manifiesto[k] for k in ["version_datos", "filas_original", "filas_limpio", "columnas_limpio", "predictoras", "huellas_sha256"]}, ensure_ascii=False, indent=1))
dic = pd.read_csv(dp.DICCIONARIO)
display(dic.groupby("rol").size().rename("columnas").to_frame())
display(dic.head(12))

## 5. Versionado

In [ ]:
r = subprocess.run(["git", "log", "--oneline", "-n", "10"], capture_output=True, text=True)
print(r.stdout or "Sin historial de git")

## 6. Visualizaciones

In [ ]:
from IPython.display import Image, display
hallazgos = vis.generar_todas(limpio)
for nombre, datos in hallazgos.items():
    print(nombre, json.dumps(datos, ensure_ascii=False, default=str))
    display(Image(filename=str(vis.DIR_FIG / nombre), width=820))

## 7. Configuración de MLflow

In [ ]:
import mlflow
uri = train.configurar_mlflow(None)
exp = mlflow.get_experiment_by_name(train.EXPERIMENTO)
print("Tracking URI:", uri); print("Experimento:", exp.name, "| id:", exp.experiment_id, "| artefactos:", exp.artifact_location)

## 8. Modelos e hiperparámetros

In [ ]:
for nombre, (est, grid) in train.definir_modelos().items():
    n = int(np.prod([len(v) for v in grid.values()]))
    print(f"{nombre:<13} {n:>3} combinaciones × 5 folds = {n*5:>3} ajustes | {grid}")

## 9. Entrenamiento y registro en MLflow

In [ ]:
import time
inicio = time.time()
res = train.ejecutar_experimento()
print(f"Tiempo: {time.time()-inicio:.0f} s | runs en el experimento: {res['runs_registrados']} | commit: {res['git_commit']}")
print(json.dumps(res["seleccion"], ensure_ascii=False, indent=1, default=str))

## 10. Resultados

In [ ]:
filas = []
for m, r in res["modelos"].items():
    filas.append({"modelo": m, "CV RMSE": r["cv_rmse"], "CV MAE": r["cv_mae"], "CV R²": r["cv_r2"],
                  "Prueba MAE": r["test_mae"], "Prueba RMSE": r["test_rmse"], "Prueba R²": r["test_r2"], "Prueba MAPE": r["test_mape"]})
tabla = pd.DataFrame(filas).set_index("modelo").sort_values("CV RMSE")
display(tabla.round(4))
display(pd.DataFrame({m: r.get("mejores_params", {}) for m, r in res["modelos"].items() if "mejores_params" in r}))

In [ ]:
anidada = pd.read_csv(RAIZ / "resultados" / "cv_anidada_resumen.csv", index_col=0)
display(anidada.round(4))
display(Image(filename=str(RAIZ / "resultados" / "figuras" / "cv_anidada_rmse.png"), width=760))

## 11. Comparación estadística

In [ ]:
pruebas = pd.read_csv(RAIZ / "resultados" / "pruebas_estadisticas.csv")
display(pruebas.round(4))
print(f"Friedman (3 modelos ajustados): χ² = {res['friedman']['estadistico']:.3f}, p = {res['friedman']['p']:.4f}")

## 12. Diagnóstico del modelo seleccionado

In [ ]:
ganador = res["seleccion"]["modelo"]
for f in [f"pred_vs_real_{ganador}.png", f"importancia_{ganador}.png"]:
    display(Image(filename=str(RAIZ / "resultados" / "figuras" / f), width=760))
print("Variables más importantes:", res["modelos"][ganador]["top_variables"])

## 13. Runs y Model Registry

In [ ]:
runs = mlflow.search_runs([exp.experiment_id])
cols = ["tags.mlflow.runName", "tags.etapa", "tags.modelo", "metrics.cv_rmse", "metrics.test_rmse", "metrics.test_r2", "tags.git_commit", "tags.version_datos"]
display(runs[[c for c in cols if c in runs.columns]].query("`tags.etapa` != 'candidato'").head(12))
print("Total de runs (incluye candidatos del grid):", len(runs))
modelo_registrado = mlflow.sklearn.load_model(f"models:/{train.NOMBRE_REGISTRO}@candidato")
df_l = pd.read_csv(dp.CSV_LIMPIO); X, y = dp.seleccionar_variables(df_l)
ganador = res["seleccion"]["modelo"]
pred_reg = modelo_registrado.predict(X.head(5))
pred_run = mlflow.sklearn.load_model(res["modelos"][ganador]["model_uri"]).predict(X.head(5))
print("Predicción del modelo registrado (MIRR/m²):", np.round(pred_reg, 3))
print("¿Idéntica a la del run?", bool(np.allclose(pred_reg, pred_run, rtol=0, atol=1e-12)))

## 14. Panel de MLflow

In [ ]:
import time, urllib.request
srv_mlflow = subprocess.Popen([sys.executable, "-m", "mlflow", "server", "--backend-store-uri", f"sqlite:///{RAIZ / 'mlflow.db'}",
                               "--host", "127.0.0.1", "--port", "5000", "--workers", "1",
                               "--allowed-hosts", "*", "--cors-allowed-origins", "*"],
                              cwd=RAIZ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=2); break
    except Exception:
        time.sleep(1)
print("Servidor de MLflow activo en el puerto 5000")
if EN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(5000, anchor_text="Abrir el panel de MLflow")

## 15. Alerta de desviación de costo

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
modelo = mlflow.sklearn.load_model(res["modelos"][ganador]["model_uri"])
estratos = pd.qcut(y, 5, labels=False)
_, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=train.SEMILLA, stratify=estratos)
razon_real = y_te / X_te["costo_prelim_m2"]; razon_est = modelo.predict(X_te) / X_te["costo_prelim_m2"]
real_alta, alerta = razon_real > 1.5, razon_est > 1.5
print(f"Obras de prueba: {len(y_te)} | con desviación real > 50 %: {int(real_alta.sum())}")
print(f"Alertas emitidas: {int(alerta.sum())} | recall {recall_score(real_alta, alerta, zero_division=0):.3f} | "
      f"precision {precision_score(real_alta, alerta, zero_division=0):.3f} | ROC-AUC de la razón estimada {roc_auc_score(real_alta, razon_est):.3f}")

## 16. Exportación de resultados

In [ ]:
import shutil, tempfile
tmp = Path(tempfile.mkdtemp()) / "resultados_colab"
for carpeta in ["resultados", "datos/datos_limp", "mlruns"]:
    if (RAIZ / carpeta).exists():
        shutil.copytree(RAIZ / carpeta, tmp / carpeta)
shutil.copy2(RAIZ / "mlflow.db", tmp / "mlflow.db")
zip_path = shutil.make_archive(str(RAIZ / "resultados_colab"), "zip", tmp)
print("Archivo:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
if EN_COLAB:
    from google.colab import files
    files.download(zip_path)